# 实验 1：从零实现基础 MLP

## 任务理解

用一个带隐藏层的多层感知机学习 XOR：两个输入不同时输出 1，相同时输出 0。训练完成后，可以在最后的“测试结果”中修改输入并查看预测。

## 理论实现

设一个 batch 有 $B$ 个样本，第 $l$ 层输入宽度为 $n_{l-1}$、输出宽度为 $n_l$。用 $L$ 表示最后的输出层，$l$ 表示中间的隐藏层。

| 符号 | 含义 | 形状 |
|---|---|---|
| $A^{(l-1)}$ | 本层输入；第 0 层就是原始数据 $X$ | $(B,n_{l-1})$ |
| $W^{(l)}$ | 本层连接权重 | $(n_{l-1},n_l)$ |
| $b^{(l)}$ | 本层偏置；对 batch 中每行广播 | $(n_l,)$ |
| $Z^{(l)}$ | 激活前的线性组合 | $(B,n_l)$ |
| $A^{(l)}$ | Sigmoid 后的激活 | $(B,n_l)$ |

前向传播只有两步：

$$
Z^{(l)}=A^{(l-1)}W^{(l)}+b^{(l)},\qquad
A^{(l)}=\sigma\!\left(Z^{(l)}\right).
$$

偏置向量等价于论文中“恒为 1 的额外输入及其连接权重”，这里只是用更直接的数组写法实现。

Sigmoid 和总误差：

$$
\sigma(z)=\frac{1}{1+e^{-z}},\qquad
E=\frac12\sum_{c,j}\left(\hat y_{c,j}-y_{c,j}\right)^2.
$$

这里对 batch **求和而不是取平均**，因此 batch 大小会影响梯度尺度。这与后续论文实验保持一致。

输出层误差信号为

$$
\delta^{(L)}
=\frac{\partial E}{\partial Z^{(L)}}
=(A^{(L)}-Y)\odot A^{(L)}\odot(1-A^{(L)}).
$$

隐藏层误差信号为

$$
\delta^{(l)}=
\left(\delta^{(l+1)}(W^{(l+1)})^\top\right)
\odot A^{(l)}\odot(1-A^{(l)}).
$$

最后把 batch 中所有样本的贡献相加：

$$
\frac{\partial E}{\partial W^{(l)}}=(A^{(l-1)})^\top\delta^{(l)},\qquad
\frac{\partial E}{\partial b^{(l)}}=\sum_c\delta_c^{(l)}.
$$

- `seed` 只固定初始权重，保证重复运行得到同样结果；
- `float64` 比 `float32` 更适合有限差分，因为数值舍入误差更小；
- `finite_difference_h=1e-5` 是数值梯度的扰动尺度：过大会偏离局部导数，过小又会被浮点舍入误差淹没；
- XOR 的 epoch、学习率和动量只用于端到端测试，不参与前面的梯度检查。

机器与 NumPy 版本被打印出来，是为了让结果具备基本的可追溯性。

In [1]:
# 环境、依赖与可复现配置
from __future__ import annotations

import platform
import sys
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

CONFIG = {
    "seed": 7,
    "dtype": "float64",
    "finite_difference_h": 1e-5,
    "gradient_abs_tolerance": 1e-8,
    "gradient_rel_tolerance": 1e-6,
    "xor_epochs": 5_000,
    "xor_learning_rate": 0.7,
    "xor_momentum": 0.9,
    "xor_loss_tolerance": 1e-3,
}

np.set_printoptions(precision=6, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

environment = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "platform": platform.platform(),
}
print("Configuration:", CONFIG)
print("Environment:", environment)


Configuration: {'seed': 7, 'dtype': 'float64', 'finite_difference_h': 1e-05, 'gradient_abs_tolerance': 1e-08, 'gradient_rel_tolerance': 1e-06, 'xor_epochs': 5000, 'xor_learning_rate': 0.7, 'xor_momentum': 0.9, 'xor_loss_tolerance': 0.001}
Environment: {'python': '3.12.14', 'numpy': '2.5.2', 'platform': 'macOS-26.6.2-arm64-arm-64bit'}


## 模型代码

下面保留最基本的前向传播、反向传播和动量更新，并用全部 4 个 XOR 样本训练网络。

In [2]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    output = np.empty_like(x)
    positive = x >= 0
    output[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    output[~positive] = exp_x / (1.0 + exp_x)
    return output


class MLP:
    def __init__(
        self,
        layer_sizes: list[int] | tuple[int, ...],
        *,
        seed: int = 0,
        init_scale: float = 0.5,
    ) -> None:
        self.layer_sizes = tuple(layer_sizes)
        rng = np.random.default_rng(seed)
        self.params: dict[str, np.ndarray] = {}
        self.grads: dict[str, np.ndarray] = {}
        self.velocity: dict[str, np.ndarray] = {}
        self.cache: list[tuple[np.ndarray, np.ndarray]] = []

        for layer, (n_in, n_out) in enumerate(
            zip(self.layer_sizes[:-1], self.layer_sizes[1:]), start=1
        ):
            self.params[f"W{layer}"] = rng.uniform(
                -init_scale, init_scale, size=(n_in, n_out)
            )
            self.params[f"b{layer}"] = rng.uniform(
                -init_scale, init_scale, size=(n_out,)
            )

        for name, parameter in self.params.items():
            self.grads[name] = np.zeros_like(parameter)
            self.velocity[name] = np.zeros_like(parameter)

    @property
    def n_layers(self) -> int:
        return len(self.layer_sizes) - 1

    def forward(self, X: np.ndarray, *, store_cache: bool = False) -> np.ndarray:
        activation = np.asarray(X, dtype=np.float64)
        if activation.ndim != 2 or activation.shape[1] != self.layer_sizes[0]:
            raise ValueError(
                f"输入形状应为 (样本数, {self.layer_sizes[0]})，实际为 {activation.shape}"
            )

        cache: list[tuple[np.ndarray, np.ndarray]] = []
        for layer in range(1, self.n_layers + 1):
            previous = activation
            activation = sigmoid(
                previous @ self.params[f"W{layer}"] + self.params[f"b{layer}"]
            )
            cache.append((previous, activation))

        if store_cache:
            self.cache = cache
        return activation

    def loss(self, X: np.ndarray, y: np.ndarray) -> float:
        prediction = self.forward(X)
        return float(0.5 * np.sum((prediction - y) ** 2))

    def backward(self, X: np.ndarray, y: np.ndarray) -> float:
        prediction = self.forward(X, store_cache=True)
        delta_from_next = prediction - y

        for layer in range(self.n_layers, 0, -1):
            previous, activation = self.cache[layer - 1]
            delta = delta_from_next * activation * (1.0 - activation)
            self.grads[f"W{layer}"] = previous.T @ delta
            self.grads[f"b{layer}"] = delta.sum(axis=0)
            delta_from_next = delta @ self.params[f"W{layer}"].T

        return float(0.5 * np.sum((prediction - y) ** 2))

    def step(self, learning_rate: float, momentum: float) -> None:
        for name in self.params:
            self.velocity[name] = (
                momentum * self.velocity[name]
                - learning_rate * self.grads[name]
            )
            self.params[name] += self.velocity[name]

In [3]:
X_train = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
], dtype=np.float64)
y_train = np.array([[0.0], [1.0], [1.0], [0.0]], dtype=np.float64)

xor_model = MLP([2, 4, 1], seed=CONFIG["seed"], init_scale=0.5)
for _ in range(CONFIG["xor_epochs"]):
    xor_model.backward(X_train, y_train)
    xor_model.step(CONFIG["xor_learning_rate"], CONFIG["xor_momentum"])

assert xor_model.loss(X_train, y_train) < CONFIG["xor_loss_tolerance"]

## 测试结果

只需修改下面的 `test_inputs`。每一行是一组 `[x1, x2]` 输入。

In [4]:
# 修改这里的输入数据
test_inputs = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
], dtype=np.float64)

if not np.all((test_inputs == 0) | (test_inputs == 1)):
    raise ValueError("测试输入只能包含 0 或 1")

expected = (test_inputs[:, 0] != test_inputs[:, 1]).astype(int)
probabilities = xor_model.forward(test_inputs).ravel()
predictions = (probabilities >= 0.5).astype(int)

print(f"训练损失: {xor_model.loss(X_train, y_train):.8f}")
print("输入 -> 正确标签 -> 输出概率 -> 预测类别")
for row, target, probability, prediction in zip(
    test_inputs, expected, probabilities, predictions
):
    print(
        f"{row.tolist()} -> {target} -> {probability:.6f} -> {prediction}"
    )

训练损失: 0.00006222
输入 -> 正确标签 -> 输出概率 -> 预测类别
[0.0, 0.0] -> 0 -> 0.005260 -> 0
[0.0, 1.0] -> 1 -> 0.994257 -> 1
[1.0, 0.0] -> 1 -> 0.994359 -> 1
[1.0, 1.0] -> 0 -> 0.005654 -> 0
